In [ ]:
# example.py
import asyncio
from nova_ai import (
    AssistantMessage,
    AssistantMessageEventStream,
    TextContent,
    ToolCall,
    Usage,
    Cost
)
from nova_ai.streaming.events import (
    StartEvent,
    TextDeltaEvent,
    DoneEvent
)

async def main():
    # 创建事件流
    stream = AssistantMessageEventStream()
    
    # 模拟异步推送事件
    async def producer():
        msg = AssistantMessage(
            model="gpt-4",
            api="openai-completions",
            provider="openai"
        )
        
        # 推送开始事件
        stream.push(StartEvent(partial=msg))
        await asyncio.sleep(3)  # 模拟网络延迟
        
        # 推送文本增量事件 - 模拟流式输出
        deltas = ["Hello", " World", "!", " How", " are", " you", "?"]
        for delta in deltas:
            stream.push(TextDeltaEvent(
                content_index=0,
                delta=delta,
                partial=msg
            ))
            await asyncio.sleep(3)  # 模拟打字间隔
        
        # 完成消息
        final_msg = AssistantMessage(
            model="gpt-4",
            api="openai-completions",
            provider="openai",
            content=[TextContent(text="Hello World! How are you?")],
            usage=Usage(
                input=10,
                output=20,
                total_tokens=30,
                cost=Cost(input=0.01, output=0.02, total=0.03)
            )
        )
        
        await asyncio.sleep(3)  # 模拟处理时间
        stream.push(DoneEvent(
            reason="stop",
            message=final_msg
        ))
        stream.close()  # 关闭流
    
    # 启动生产者
    asyncio.create_task(producer())
    print(stream)
    # 消费事件
    print("开始接收事件流...\n")
    async for event in stream:
        print(f"事件类型: {event.type}")
        if event.type == "start":
            print(f"  开始生成回复...")
        elif event.type == "text_delta":
            print(f"  增量: '{event.delta}'")
            print(f"  当前内容: {event.partial.content}")
        elif event.type == "done":
            print(f"\n完成原因: {event.reason}")
            print(f"最终消息: {event.message.content[0].text}")
            print(f"使用统计: {event.message.usage}")
    
    # 获取最终结果
    result = await stream.result()
    print(f"\n最终结果: {result}")

if __name__ == "__main__":
    await main()

In [ ]:
"""
code_assembler.py - 代码文件组装工具
将指定目录下的代码文件组装成一个Markdown文件，便于提交给大模型
"""

import os
import argparse
from pathlib import Path
from typing import List, Set, Optional
import fnmatch

class CodeAssembler:
    """代码文件组装器"""
    
    # 默认忽略的目录和文件
    DEFAULT_IGNORE_DIRS = {
        '__pycache__', '.git', '.svn', '.hg', 
        'node_modules', 'venv', 'env', '.env',
        'dist', 'build', '.idea', '.vscode'
    }
    
    DEFAULT_IGNORE_FILES = {
        '*.pyc', '*.pyo', '*.so', '*.dll', '*.dylib',
        '*.class', '*.o', '*.obj', '*.exe', '*.bin',
        '*.jpg', '*.jpeg', '*.png', '*.gif', '*.bmp',
        '*.mp3', '*.mp4', '*.avi', '*.mov',
        '*.zip', '*.tar', '*.gz', '*.rar', '*.7z',
        '*.pdf', '*.doc', '*.docx', '*.xls', '*.xlsx'
    }
    
    # 常见的代码文件扩展名
    CODE_EXTENSIONS = {
        '.py', '.js', '.jsx', '.ts', '.tsx', '.java', '.c', '.cpp',
        '.h', '.hpp', '.cs', '.go', '.rb', '.php', '.swift', '.kt',
        '.rs', '.scala', '.html', '.css', '.scss', '.less', '.sql',
        '.sh', '.bash', '.zsh', '.fish', '.ps1', '.bat', '.cmd',
        '.json', '.yaml', '.yml', '.xml', '.toml', '.ini', '.conf',
        '.md', '.rst', '.txt', '.csv', '.tsv'
    }
    
    def __init__(self, 
                 root_path: str,
                 output_file: str = "code_assembly.md",
                 ignore_dirs: Optional[Set[str]] = None,
                 ignore_files: Optional[Set[str]] = None,
                 include_extensions: Optional[Set[str]] = None,
                 max_file_size: int = 10 * 1024 * 1024,  # 默认10MB
                 include_hidden: bool = False):
        """
        初始化代码组装器
        
        Args:
            root_path: 要读取的根目录路径
            output_file: 输出Markdown文件路径
            ignore_dirs: 要忽略的目录名集合（支持通配符）
            ignore_files: 要忽略的文件名模式集合（支持通配符）
            include_extensions: 要包含的文件扩展名集合
            max_file_size: 最大文件大小（字节）
            include_hidden: 是否包含隐藏文件
        """
        self.root_path = Path(root_path).resolve()
        self.output_file = output_file
        self.ignore_dirs = ignore_dirs or self.DEFAULT_IGNORE_DIRS.copy()
        self.ignore_files = ignore_files or self.DEFAULT_IGNORE_FILES.copy()
        self.include_extensions = include_extensions or self.CODE_EXTENSIONS.copy()
        self.max_file_size = max_file_size
        self.include_hidden = include_hidden
        
    def should_ignore_dir(self, dir_name: str) -> bool:
        """检查目录是否应该被忽略"""
        for pattern in self.ignore_dirs:
            if fnmatch.fnmatch(dir_name, pattern):
                return True
        return False
    
    def should_ignore_file(self, file_name: str) -> bool:
        """检查文件是否应该被忽略"""
        # 检查隐藏文件
        if not self.include_hidden and file_name.startswith('.'):
            return True
            
        # 检查文件扩展名
        ext = Path(file_name).suffix.lower()
        if ext not in self.include_extensions:
            return True
            
        # 检查文件名模式
        for pattern in self.ignore_files:
            if fnmatch.fnmatch(file_name, pattern):
                return True
                
        return False
    
    def get_file_language(self, file_path: Path) -> str:
        """根据文件扩展名获取代码语言"""
        ext = file_path.suffix.lower()
        language_map = {
            '.py': 'python',
            '.js': 'javascript',
            '.jsx': 'javascript',
            '.ts': 'typescript',
            '.tsx': 'typescript',
            '.java': 'java',
            '.c': 'c',
            '.cpp': 'cpp',
            '.h': 'c',
            '.hpp': 'cpp',
            '.cs': 'csharp',
            '.go': 'go',
            '.rb': 'ruby',
            '.php': 'php',
            '.swift': 'swift',
            '.kt': 'kotlin',
            '.rs': 'rust',
            '.scala': 'scala',
            '.html': 'html',
            '.css': 'css',
            '.scss': 'scss',
            '.sql': 'sql',
            '.sh': 'bash',
            '.bash': 'bash',
            '.json': 'json',
            '.yaml': 'yaml',
            '.yml': 'yaml',
            '.xml': 'xml',
            '.toml': 'toml',
            '.ini': 'ini',
            '.md': 'markdown',
        }
        return language_map.get(ext, '')
    
    def read_file_content(self, file_path: Path) -> Optional[str]:
        """读取文件内容"""
        try:
            # 检查文件大小
            file_size = file_path.stat().st_size
            if file_size > self.max_file_size:
                return f"[文件过大，跳过：{file_size} 字节 > {self.max_file_size} 字节]"
            
            # 尝试不同的编码读取文件
            encodings = ['utf-8', 'gbk', 'gb2312', 'latin-1']
            for encoding in encodings:
                try:
                    with open(file_path, 'r', encoding=encoding) as f:
                        return f.read()
                except UnicodeDecodeError:
                    continue
                    
            return f"[无法解码文件，尝试的编码：{', '.join(encodings)}]"
            
        except Exception as e:
            return f"[读取文件失败：{str(e)}]"
    
    def collect_files(self) -> List[Path]:
        """收集所有符合条件的文件"""
        files = []
        
        for root, dirs, filenames in os.walk(self.root_path):
            # 过滤目录
            dirs[:] = [d for d in dirs if not self.should_ignore_dir(d)]
            
            for filename in filenames:
                file_path = Path(root) / filename
                
                # 检查是否应该忽略
                if self.should_ignore_file(filename):
                    continue
                
                # 获取相对路径
                rel_path = file_path.relative_to(self.root_path)
                files.append(file_path)
                
        # 按路径排序
        files.sort(key=lambda p: str(p))
        return files
    
    def assemble(self) -> str:
        """
        组装代码文件为Markdown格式
        
        Returns:
            生成的Markdown内容
        """
        files = self.collect_files()
        
        # 生成项目结构
        tree_structure = self.generate_tree_structure()
        
        # 构建Markdown内容
        md_lines = []
        md_lines.append("# 代码文件汇总\n")
        md_lines.append(f"- **项目根目录**: `{self.root_path}`")
        md_lines.append(f"- **文件总数**: {len(files)}")
        md_lines.append(f"- **生成时间**: {self._get_current_time()}\n")
        
        md_lines.append("## 目录结构\n")
        md_lines.append("```")
        md_lines.append(tree_structure)
        md_lines.append("```\n")
        
        md_lines.append("## 文件内容\n")
        
        for i, file_path in enumerate(files, 1):
            rel_path = file_path.relative_to(self.root_path)
            content = self.read_file_content(file_path)
            language = self.get_file_language(file_path)
            
            md_lines.append(f"### {i}. {rel_path}\n")
            md_lines.append(f"**路径**: `{rel_path}`\n")
            md_lines.append(f"**大小**: {self._format_size(file_path.stat().st_size)}\n")
            
            if content:
                md_lines.append(f"```{language}")
                md_lines.append(content.rstrip())
                md_lines.append("```\n")
            else:
                md_lines.append("*[文件内容为空]*\n")
        
        # 写入文件
        output_path = Path(self.output_file)
        output_path.write_text('\n'.join(md_lines), encoding='utf-8')
        
        return '\n'.join(md_lines)
    
    def generate_tree_structure(self) -> str:
        """生成目录树结构"""
        tree_lines = []
        
        def add_to_tree(path: Path, prefix: str = "", is_last: bool = True):
            if path == self.root_path:
                tree_lines.append(f"{self.root_path.name}/")
            else:
                rel_path = path.relative_to(self.root_path)
                name = path.name
                
                if path.is_dir():
                    tree_lines.append(f"{prefix}└── {name}/" if is_last else f"{prefix}├── {name}/")
                    prefix += "    " if is_last else "│   "
                    
                    # 获取子项目
                    items = sorted(path.iterdir(), key=lambda p: (p.is_file(), p.name))
                    items = [item for item in items 
                            if not self.should_ignore_dir(item.name) if item.is_dir()
                            or not self.should_ignore_file(item.name) if item.is_file()]
                    
                    for i, item in enumerate(items):
                        is_last_item = i == len(items) - 1
                        add_to_tree(item, prefix, is_last_item)
                else:
                    tree_lines.append(f"{prefix}└── {name}" if is_last else f"{prefix}├── {name}")
        
        add_to_tree(self.root_path)
        return '\n'.join(tree_lines)
    
    def _get_current_time(self) -> str:
        """获取当前时间字符串"""
        from datetime import datetime
        return datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    def _format_size(self, size: int) -> str:
        """格式化文件大小"""
        for unit in ['B', 'KB', 'MB', 'GB']:
            if size < 1024:
                return f"{size:.1f} {unit}"
            size /= 1024
        return f"{size:.1f} TB"


def main():
    """命令行入口"""
    # 创建组装器并执行
    assembler = CodeAssembler(
        root_path='/root/nova/packages/nova_ai/src/nova_ai',
        output_file='/root/nova/packages/nova_ai/src/output.md',
        include_hidden=False,
    )
    
    assembler.assemble()
    print(f"✅ 成功生成代码汇总文件：'/root/nova/packages/nova_ai/src'")

In [ ]:
main()

In [3]:
# simple_example.py
import asyncio
import os
# os.environ["HTTP_PROXY"] = "http://10.0.1.158:8118"
# os.environ["HTTPS_PROXY"] = "http://10.0.1.158:8118"
from nova_ai.core.messages import Tool
from nova_ai import (
    get_model, Context, UserMessage,
    stream_simple,
    SimpleStreamOptions,
    OpenAICompletionsOptions, get_env_api_key
)
class WeatherParameters:
    """天气查询参数"""
    type = "object"
    properties = {
        "city": {
            "type": "string",
            "description": "城市名称，例如：北京、上海、广州"
        },
        "date": {
            "type": "string",
            "description": "查询日期，格式：YYYY-MM-DD，默认为今天"
        },
        "units": {
            "type": "string",
            "enum": ["celsius", "fahrenheit"],
            "description": "温度单位，默认为摄氏度"
        }
    }
    required = ["city"]
async def simple_chat():
    """最简单的使用示例"""
    
    # 1. 获取模型
    model = get_model("volcengine", "deepseek-r1-250528")
    print(model)
    # 2. 创建上下文
    
    context = Context(
        messages=[
            UserMessage(content="你好，查询北京天气")
        ],
        tools=[
            Tool(
                name="get_weather",
                description="查询指定城市的天气信息，包括温度、湿度、风速和天气状况",
                parameters={
                    "type": "object",
                    "properties": {
                        "city": {
                            "type": "string",
                            "description": "城市名称，例如：北京、上海、广州"
                        },
                        "date": {
                            "type": "string",
                            "description": "查询日期，格式：YYYY-MM-DD，默认为今天",
                            "pattern": "^\\d{4}-\\d{2}-\\d{2}$"  # 添加正则验证
                        },
                        "units": {
                            "type": "string",
                            "enum": ["celsius", "fahrenheit"],
                            "description": "温度单位，默认为摄氏度",
                            "default": "celsius"
                        }
                    },
                    "required": ["city"]
                }
            )
        ]
    )
    
    # 3. 创建选项
    options = SimpleStreamOptions(
        api_key="3b631f71-6bd6-464a-9abc-b0e8d19f25d7",
        max_tokens=10000,
    )
    print(type(options))
    # 4. 流式调用
    print("助手: ", end="")
    stream1 = await stream_simple(model, context, options)
    print(stream1)
    async for event in stream1:
        print(event)
    
    print("\n")

# 运行
await simple_chat()

Model(id='deepseek-r1-250528', name='Deepseek-R1', api=<KnownApi.OPENAI_COMPLETIONS: 'openai-completions'>, provider=<KnownProvider.VOLCENGINE: 'volcengine'>, base_url='https://ark.cn-beijing.volces.com/api/v3/', reasoning=True, input_types=['text'], cost=ModelCost(input=2.0, output=8.0, cache_read=0.5, cache_write=0.0), context_window=1047576, max_tokens=32768, headers=None, compat=OpenAICompletionsCompat(supports_store=None, supports_developer_role=None, supports_reasoning_effort=None, supports_usage_in_streaming=None, max_tokens_field=None, requires_tool_result_name=None, requires_assistant_after_tool_result=None, requires_thinking_as_text=None, requires_mistral_tool_ids=None, thinking_format=<ThinkingFormat.OPENAI: 'openai'>, supports_strict_mode=None, open_router_routing=None, vercel_gateway_routing=None))
<class 'nova_ai.utils.stream_options.SimpleStreamOptions'>
助手: <nova_ai.streaming.event_stream.AssistantMessageEventStream object at 0x7fac1ab3d1d0>
StartEvent(type='start', par

In [ ]:
<nova_ai.streaming.event_stream.AssistantMessageEventStream object at 0x7fed192d7bd0>

In [ ]:
from openai.types.chat import ChatCompletionContentPart